# 迭代器与生成器
**引言**: 迭代器和生成器是Python的“内存优化神器I-能轻松解决“大数据占满内存”的难题，比如处理100万条数据时，无需像列表那样一次性加载全部，而是按需生成、即用即释。虽然涉及惰性计算的核心逻辑，但本质是“高效利用内存”的设计思路。掌握这两个特性，能摆脱内存限制的束缚，轻松应对超大文件、流式数据等场景，是处理大数据的必备技能。

---

# 1. 可迭代对象（能被 for 循环遍历的对象）

## 1.1. 什么是可迭代对象？

可迭代对象就像“装满东西的容器”，比如装满苹果的篮子、写满文字的笔记本，你可以一个个拿出里面的东西（遍历）。

## 1.2. 常见的可迭代对象

- 内置数据类型：字符串（str）、列表（list）、元组（tuple）、字典（dict）、集合（set）、range 序列；
- 其他：文件对象、迭代器、生成器。

## 1.3. 如何判断一个对象是否可迭代?
用`isinstance()`结合`collections.abc.Iterable` 类,就能快速判断。


In [2]:
from collections.abc import Iterable

print(isinstance([1,2,3], Iterable))  # True
print(isinstance((1,2,3), Iterable))  # True
print(isinstance({1,2,3}, Iterable))  # True
print(isinstance({'a':1, 'b':2}, Iterable))  # True
print(isinstance(123, Iterable))  # False 数字不是可迭代对象
print(isinstance("Hello", Iterable))  # True

True
True
True
True
False
True


---

# 2. 迭代器：记录遍历位置的“指针”

## 2.1. 什么是迭代器？

迭代器是可迭代对象的“底层驱动”——它就像一个“指针”，记住当前遍历到哪个元素，每次调用它，就返回下一个元素，直到所有元素遍历完。

生活案例：你用遥控器换台，遥控器就是“迭代器”，按一次“下一个”（next()），就切换到下一个频道（元素），直到所有频道都看完（抛出异常，停止遍历）。

## 2.2. 迭代器的 2 个核心方法

| 方法               | 作用                         | 底层调用                         |
|--------------------|------------------------------|----------------------------------|
| `iter(可迭代对象)`   | 获取可迭代对象的迭代器       | 调用对象的 `__iter__` 方法         |
| `next(迭代器)`       | 获取迭代器的下一个元素       | 调用迭代器的 `__next__` 方法       |

In [4]:
li = [1, 2, 3]
# 通过iter()函数获取迭代器对象
it = iter(li)
print(it)  # <list_iterator object at 0x7f8c8c

# 通过next()函数获取下一个元素
print(next(it))  # 1
print(next(it))  # 2
print(next(it))  # 3

1
2
3


## 2.3. for 循环的工作原理

我们平时用 `for` 循环遍历可迭代对象，底层其实是自动做了 3 件事：

1. 调用 `iter(可迭代对象)`，获取迭代器；
2. 不断调用 `next(迭代器)`，获取下一个元素，赋值给循环变量（如 i）；
3. 当遍历完所有元素，`next()` 会抛出 `StopIteration` 异常，`for` 循环自动捕获这个异常，结束循环。

简单说：`for` 循环是迭代器的“语法糖”，帮我们省去了手动调用 `next()` 和处理异常的麻烦！

## 2.4. 可迭代对象 vs 迭代器（核心区别）

| 对比维度   | 可迭代对象（如列表、字符串）           | 迭代器（如 `iter(列表)` 返回的对象）         |
|------------|----------------------------------------|--------------------------------------------|
| 遍历方式   | 只能用 `for` 循环遍历                    | 可用 `next()` 逐个获取，也可用 `for` 循环       |
| 核心方法   | 有`__iter__ `方法（返回迭代器）         | 有 `__iter__` 方法（返回自身）和 `__next__` 方法 |
| 内存占用   | 一次性加载所有数据到内存               | 按需生成数据，不占额外内存                 |
| 可重复使用 | 可以重复遍历（每次 `for` 循环都重新获取迭代器） | 一次性的，遍历完后不能再用（需重新创建）     |

#### 关键提醒

如果内置迭代器满足不了需求，也可以自己写迭代器类，核心是实现 __iter__ 和 __next__ 方法。

In [8]:
# 自定义一个迭代器，生成1-5的数字
class MyIterator:
    def __init__(self):
        self.current = 1

    def __iter__(self):
        return self

    def __next__(self):
        if self.current > 5:
            raise StopIteration
        else:
            num = self.current
            self.current += 1
            return num
        
my_iter = MyIterator()
for num in my_iter:
    print(num)  # 输出1-5

for num in my_iter:             # 迭代器已经迭代完了，再次迭代会直接停止（一次性）
    print(num)  # 输出1-5

1
2
3
4
5


---
# 3. 生成器：特殊的迭代器（内存优化神器）

## 3.1. 什么是生成器？

生成器是“自带迭代器功能的特殊函数 / 表达式”，本质是迭代器，但比手动写迭代器简单 10 倍！它的核心是“惰性计算”——不用提前生成所有数据，用到的时候才生成，用完就释放，极大节省内存。

生活案例：面包店不一次性烤 1000 个面包（占仓库），而是顾客来一个烤一个（按需生成），这就是生成器的思路。

## 3.2. 为什么需要生成器？

- **列表的痛点**：处理 100 万条数据时，[i for i in range(1000000)] 会一次性加载所有数据到内存，导致内存爆满；
- **生成器的优势**：(i for i in range(1000000))不会加载任何数据到内存，只有调用时才生成下一个元素，内存占用几乎可以忽略。

## 3.3. 生成器的 2 种定义方式（重点！）
#### 方式 1：生成器表达式（最简单）

把列表推导式的 `[]` 改成 `()`，就是生成器表达式，语法简单。

In [10]:
# 列表推导式：[表达式 for 变量 in 可迭代对象 if 条件]
print([i for i in range(10)])

print([i*5 for i in range(10) if i % 2 == 0])  # 输出偶数的5倍


[0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
[0, 10, 20, 30, 40]


#### 方式 2：生成器函数（带`yield`关键字）
在函数体内加入 `yield` 关键字，这个函数就变成了生成器函数——调用时不会执行函数体，而是返回一个生成器对象，只有用 `next()` 或 `for` 循环触发，才会执行函数体，遇到 `yield` 就返回值并暂停。

`yield` 的核心作用：**返回值 + 暂停函数 + 保存状态**（和 `return` 的区别：`return` 会终止函数，`yield` 只是暂停）。

In [12]:
# 写一个生成器函数 逐步返回水果名称

def fruit_generator():
    print("开始生成水果...")
    yield "苹果"
    print("生成了一个水果...")
    yield "香蕉"
    print("生成了一个水果...")
    yield "橙子"

for fruit in fruit_generator():
    print(fruit)

开始生成水果...
苹果
生成了一个水果...
香蕉
生成了一个水果...
橙子


## 3.4. 生成器的核心优势

1. 节省内存：按需生成数据，不一次性加载，适合处理大数据。
2. 简化代码：不用手动实现 `__iter__` 和 `__next__` 方法，比自定义迭代器简单。
3. 支持暂停恢复：yield 能保存函数状态，下次调用从暂停处继续，适合分步处理任务。

## 4. 核心概念对比：可迭代对象 vs 迭代器 vs 生成器

| 概念       | 核心定义                     | 关键特征                          | 内存占用                     | 适用场景                         |
|------------|------------------------------|-----------------------------------|------------------------------|----------------------------------|
| 可迭代对象 | 能被 for 循环遍历的对象      | 有 `__iter__` 方法，返回迭代器       | 一次性加载所有数据（如列表） | 数据量小，需要重复遍历           |
| 迭代器     | 记录遍历位置的对象           | 有 `__iter__` 和 `__next__` 方法       | 按需生成，不占额外内存       | 自定义遍历逻辑，一次性遍历       |
| 生成器     | 特殊的迭代器（惰性计算）     | 用 `()` 或 `yield` 定义               | 几乎不占内存                 | 大数据处理，分步生成数据         |

可迭代对象>迭代器>生成器  
- 生成器一定是迭代器，迭代器一定是可迭代对象
- 可迭代对象不一定是迭代器（如列表），迭代器不一定是生成器（如自定义迭代器类）

# 5. 核心总结

1. 可迭代对象：能被 `for` 循环遍历，核心是有 `__iter__` 方法；
2. 迭代器：能通过 `next()` 逐个取值，核心是有 `__next__` 方法，一次性遍历，省内存；
3. 生成器：特殊迭代器，核心是“惰性计算”，定义方式有 2 种：
   - 生成器表达式：（表达式 `for 变量 in 可迭代对象`）；
   - 生成器函数：带 yield 关键字的函数，调用返回生成器对象；
4. 核心价值：处理大数据时，用生成器替代列表，避免内存爆满；
5. 实战技巧：
   - 数据量小（<10000）：用列表（方便重复遍历）；
   - 数据量大（>10000）：用生成器（省内存）；
   - 自定义遍历逻辑：用迭代器；
   - 分步生成数据：用生成器函数（`yield` 暂停恢复）。